This notebook is served as the code of Appendix C.4 "Cascading bandit".

1. Experiment Name: Comparison experiment under descending-order cascading feedback;
2. Dataset: synthetic;
3. Range: $[0.3,0.4]$;
4. Description: In this notebook, we fix $k=10, m=30$ and $k=20, m=30$ in the descending-order cascading feedback setting.

In [ ]:
import numpy as np
import os
import time
import pandas as pd
from tqdm import tqdm
import random

GLOBAL_SEED = 2026
# -------- Bandit Class --------
class PerturbedBernoulliBandit:
    def __init__(self, mu, noise_scale):
        self.base_mu = np.clip(mu, 0.3, 0.4)
        self.noise_scale = noise_scale
    
    def pull(self):
        perturbed_mu = np.clip(
            self.base_mu + np.random.normal(0, self.noise_scale),
            0.3, 0.4
        )
        return np.random.binomial(1, perturbed_mu)

# -------- our proposed CMOSS algorithm --------  
class CMOSS:
    def __init__(self, m, k, delta):
        self.m = m
        self.k = k
        self.delta = delta
        self.T_i = np.zeros(m)          
        self.hat_mu = np.ones(m)       
    
    def select_arms(self):
        unexplored = np.where(self.T_i == 0)[0]
        if len(unexplored) >= self.k:
            return unexplored[:self.k]
        T_safe = np.where(self.T_i == 0, 1e-8, self.T_i)  
        log_input = 1 / (self.delta * T_safe)
        log_input = np.clip(log_input, 1.0, 1e100)        
        rho = np.sqrt(np.log(log_input) / T_safe)
        rho = np.nan_to_num(rho, nan=0.0, posinf=0.0, neginf=0.0)
        bar_mu = np.minimum(self.hat_mu + rho, 1.0)       
        return np.argsort(bar_mu)[-self.k:][::-1]
    
    def update(self, selected, rewards):
        for i, arm in enumerate(selected):
            self.T_i[arm] += 1
            self.hat_mu[arm] += (rewards[i] - self.hat_mu[arm]) / self.T_i[arm]

# -------- CUCB algorithm --------
class CUCB:
    def __init__(self, m, k):
        self.m = m
        self.k = k
        self.T_i = np.zeros(m)       
        self.hat_mu = np.ones(m)     
        self.t = 0                   
    
    def select_arms(self):
        self.t += 1                   
        unexplored = np.where(self.T_i == 0)[0]
        if len(unexplored) >= self.k:
            return unexplored[:self.k]
        T_safe = np.where(self.T_i == 0, np.inf, self.T_i)  
        rho = np.sqrt(3 * np.log(self.t) / (2 * T_safe))    
        bar_mu = np.minimum(self.hat_mu + rho, 1.0)        
        return np.argsort(bar_mu)[-self.k:][::-1]
    
    def update(self, selected, rewards):
        for i, arm in enumerate(selected):    
            self.T_i[arm] += 1
            self.hat_mu[arm] += (rewards[i] - self.hat_mu[arm]) / self.T_i[arm]

# -------- comparison under descending-order cascading feedback --------  
class ExperimentRunner:
    def __init__(self, noise_scale):
        self.noise_scale = noise_scale
        self.datasets = {
            'synthetic_30arms': np.random.uniform(0.3, 0.4, 30).tolist(),
        }
        self.param_grid = {
            'CMOSS': {'k': [10, 20], 'delta': [0.00001]},
            'CUCB': {'k': [10, 20]},
        }
        self.data_dir = "results"
        os.makedirs(self.data_dir, exist_ok=True)

    def _cascading_reward(self, arms, bandits):
        """Compute expected reward under cascading model."""
        no_click_prob = 1.0
        for arm_idx in arms:
            no_click_prob *= (1 - bandits[arm_idx].base_mu)
        return 1 - no_click_prob

    def run(self, n_sim, T):
        conv_data = []

        for dataset, mus in self.datasets.items():
            sorted_mus = sorted(mus, reverse=True)
            for algo in ['CMOSS', 'CUCB']:
                params_list = self._gen_params(algo)
                for params in params_list:
                    k = params['k']
                    best_exp = 1 - np.prod([1 - mu for mu in sorted_mus[:k]])  # Optimal expected reward

                    for sim_id in tqdm(range(n_sim), desc=f"{dataset}-{algo}-k{k}"):
                        sim_seed = GLOBAL_SEED + sim_id * 1000 + hash((algo, k)) % 10_000_000
                        random.seed(sim_seed)
                        np.random.seed(sim_seed)
                            
                        bandits = [PerturbedBernoulliBandit(mu, self.noise_scale) for mu in mus]
                        agent = self._init_algo(algo, len(mus), params)
                  
                        cumulative_time = 0.0
                        current_regret = 0.0

                        regrets = []
                        times = []

                        for t in range(T):
                            step_start = time.perf_counter()
                            arms = agent.select_arms()

                            # Simulate cascading click feedback
                            rewards = [0] * k
                            clicked_index = -1
                            for pos, arm_idx in enumerate(arms):
                                if bandits[arm_idx].pull() == 1:
                                    rewards[pos] = 1
                                    clicked_index = pos
                                    break  # Cascade ends after first click

                            observed_rewards = [0] * k
                            for i in range(k):
                                if i == clicked_index:
                                    observed_rewards[i] = 1
                                    break
                                else:
                                    observed_rewards[i] = 0
                            agent.update(arms[:clicked_index+1] if clicked_index >= 0 else arms, 
                                         observed_rewards[:clicked_index+1] if clicked_index >= 0 else observed_rewards)

                            # Regret computation
                            current_reward = 1 - np.prod([1 - bandits[i].base_mu for i in arms])
                            current_regret += best_exp - current_reward
                            regrets.append(current_regret)

                            step_time = time.perf_counter() - step_start
                            cumulative_time += step_time
                            times.append(cumulative_time)

                            # Store final data
                            conv_data.append({
                                "algorithm": algo,
                                "dataset": dataset,
                                "params": params,
                                "sim_id": sim_id,
                                "k": k,
                                "t": t + 1,
                                'noise': self.noise_scale,
                                "regret": current_regret,
                                "cumulative_time": cumulative_time
                            })                        
        return pd.DataFrame(conv_data)
    
    def save_data(self, conv_df):
        conv_df.to_csv(f"{self.data_dir}/s_conv_syn.csv", index=False)

    def _gen_params(self, algo):
        params = [] 
        if algo == 'CMOSS':
            for k in self.param_grid[algo]['k']:
                for delta in self.param_grid[algo]['delta']:
                    params.append({'k': k, 'delta': delta})
        else:
            for k in self.param_grid[algo]['k']:
                params.append({'k': k})
        return params

    def _init_algo(self, algo, m, params):
        if algo == 'CMOSS':
            return CMOSS(m=m, k=params['k'], delta=params['delta'])
        elif algo == 'CUCB':
            return CUCB(m=m, k=params['k'])

        
if __name__ == "__main__":
    exp = ExperimentRunner(noise_scale = 0)
    n_sim, T = 10, 100000
    conv_df = exp.run(n_sim, T)
    exp.save_data(conv_df) 

synthetic_30arms-CUCB-k20: 100%|██████████| 10/10 [00:46<00:00,  4.64s/it]


In [2]:
def process_results(input_path, output_path, target_steps, target_k=[10, 20]):
    df = pd.read_csv(input_path)

    if "k" in df.columns:
        df = df[df["k"].isin(target_k)]
    else:
        raise ValueError("Input CSV does not contain column `k`, cannot split by k.")

    filtered = df[df['t'].isin(target_steps)]

    grouped = filtered.groupby(['algorithm', 'k', 't']).agg({
        'regret': 'mean',
        'cumulative_time': 'mean'
    }).reset_index()
    
    result = grouped.melt(
        id_vars=['algorithm', 'k', 't'],
        value_vars=['regret', 'cumulative_time'],
        var_name='metric',
        value_name='average_value'
    )
    
    result.to_csv(output_path, index=False)
    print(f"Processed results saved to {output_path}")

if __name__ == "__main__":
    config = {
        'input_file': r'results/s_conv_syn.csv',
        'output_file': r'results/s_summary_results_syn.csv',
        'target_steps': [100000],
        'target_k': [10, 20]
    }
    
    process_results(
        input_path=config['input_file'],
        output_path=config['output_file'],
        target_steps=config['target_steps'],
        target_k=config['target_k']
    )


Processed results saved to results/s_summary_results_syn.csv


In [3]:
df = pd.read_csv("results/s_conv_syn.csv")

df = df[df["k"].isin([10, 20])]
last_steps = df[df["t"] == df["t"].max()]

result = (
    last_steps
    .groupby(["algorithm", "k"])["cumulative_time"]
    .mean()
    .reset_index()
)

result["Average Time per Step (sec)"] = result["cumulative_time"] / df["t"].max()
result = result.drop(columns="cumulative_time")
result = result.rename(columns={
    "algorithm": "Algorithm",
    "k": "k"
})

print("Time per Step by k")
print(result.to_markdown(index=False, floatfmt=".8f"))


Time per Step by k
| Algorithm   |   k |   Average Time per Step (sec) |
|:------------|----:|------------------------------:|
| CMOSS       |  10 |                    0.00005012 |
| CMOSS       |  20 |                    0.00007286 |
| CUCB        |  10 |                    0.00003328 |
| CUCB        |  20 |                    0.00004576 |
